# 6. ResNet-50 on the combined dataset (donor-grouped CV)

ResNet-50 trained on the combined dataset (847 expert labels plus 4,834 machine labels)
with 5-fold cross-validation grouped by donor, on the same fold partition as the
human-only run, scored only on the 847 expert labels. Together with notebook 05 this
completes the 2x2 grid of architecture by training data.

Two runs of this configuration exist. The one executed in this notebook is a 448px
reproduction on a 6 GB laptop GPU (`models/quality_resnet_cvF/`). The 640px run on the
university GPU server (`models/quality_resnet_cvF640/`) is the one quoted in the
dissertation and in notebook 05's grid. The difference between them is the resolution
effect.

`DEMO = True` trains fold 1 live. `FULL_RUN = True` runs all five folds and skips any fold
already finished. With both `False` the notebook is offline and shows stored results.
Evaluation is identical to the human-only run. Training uses the noise-aware recipe
(soft targets, balanced sampler, geometric augmentation only, then a fine-tune on the
expert labels).


In [1]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from nb_style import (ROOT, p, TRAIN_COLOR, VAL_COLOR, set_seed,
                      metrics_report, verify, verify_summary)

DEMO        = False   # True -> train fold 1 live below (448px, fits a 6 GB laptop GPU)
FULL_RUN    = True    # stored run: 448px reproduction on the laptop GPU, 2026-09-01 (640px run lives in quality_resnet_cvF640)
GPU         = "0"    # CUDA device index on this machine (on garlick pick an idle one, e.g. "5")
DEMO_EPOCHS = 6

os.environ["CUDA_VISIBLE_DEVICES"] = GPU
# dataset + recipe env MUST be set before the engine import (train_quality_cnn reads env at import)
os.environ["QUALITY_CSV"] = p("review_tool", "results", "combined_human_s2usable_20260828.csv")
_local = str(ROOT)   # laptop: images live in the MAIN checkout
if os.path.isdir(os.path.join(_local, "data_bulk", "images")):
    os.environ["QUALITY_IMG_DIRS"] = os.pathsep.join([
        os.path.join(_local, "review_tool", "webapp", "review_images"),
        os.path.join(_local, "data_bulk", "images")])
else:                                     # garlick: repo-relative layout
    os.environ["QUALITY_IMG_DIRS"] = os.pathsep.join(
        [p("review_tool", "webapp", "review_images"), p("data_bulk", "images")])
os.environ["QUALITY_INPUT_SIZE"] = "448"   # local card/RAM limit; 640 on garlick - caveat recorded
os.environ["QUALITY_BATCH"] = "12"
os.environ["QUALITY_SOFT_TARGETS"] = "1"   # scope-03c noise-aware recipe (Arm D)
os.environ["QUALITY_SAMPLER"] = "1"
os.environ["QUALITY_AUG"] = "geo"
os.environ["QUALITY_TWO_STAGE"] = "1"
os.environ["RESF_OUT"] = p("models", "quality_resnet_cvF")

sys.path.insert(0, ROOT)
import resnet_cvF as rf            # imports train_quality_cnn with the env above

set_seed(rf.tq.SEED)
print(f"engine: resnet_cvF.py reusing train_quality_cnn.py  (seed {rf.tq.SEED})")
print(f"input {rf.tq.INPUT_SIZE}px  batch {rf.tq.BATCH}  max epochs {rf.tq.EPOCHS}")
print(f"recipe: soft_targets={rf.tq.SOFT_TARGETS} sampler={rf.tq.USE_SAMPLER} "
      f"geo_aug={rf.tq.GEO_AUG_ONLY} two_stage={rf.tq.TWO_STAGE}")
%matplotlib inline
# ^ restores inline figures (train_quality_cnn forces the Agg backend at import)


engine: resnet_cvF.py reusing train_quality_cnn.py  (seed 42)
input 448px  batch 12  max epochs 20
recipe: soft_targets=True sampler=True geo_aug=True two_stage=True


## 1. The combined dataset

5,681 slides: 847 expert labels plus 4,834 machine labels kept after the vote-confidence
filter. The unusable class is only 6.0%, so the balanced sampler matters.


In [2]:
c = rf.census()
print(f"dataset: {c['csv']}")
print(f"rows {c['rows']}   human {c['human']}   machine {c['machine']}   donors {c['donors']}")
print(f"labels: usable {c['usable']}   unusable {c['unusable']} "
      f"({c['unusable'] / c['rows'] * 100:.1f}%)")

dataset: combined_human_s2usable_20260828.csv
rows 5681   human 847   machine 4834   donors 43
labels: usable 5341   unusable 340 (6.0%)


## 2. The fold partition

Test donors per fold are read from the human-only run, so both runs are tested on the
identical slides. All slides of a fold's test donors, human and machine, are kept out of
its training set. The assert below checks this.


In [3]:
part = rf.fold_partition()
print(f"{'fold':>4s} {'test donors':>12s} {'test human rows':>16s} {'train rows':>11s} "
      f"{'ArmB F1':>8s}")
for f in part:
    print(f"{f['fold']:4d} {len(f['test_donors']):12d} {f['n_test_human']:16d} "
          f"{f['n_train_rows']:11d} {f['armB_macro_f1']:8.4f}")
print(f"\nsum of test rows: {sum(f['n_test_human'] for f in part)}  (= all 847 expert labels, once each)")

fold  test donors  test human rows  train rows  ArmB F1
   1            8              148        4558   0.7876
   2            9              220        4509   0.7765
   3           10              153        4553   0.8341
   4            8              151        4532   0.8042
   5            8              175        4572   0.8183

sum of test rows: 847  (= all 847 expert labels, once each)


## 3. Live training (DEMO, fold 1)

Set `DEMO = True` in the config cell to train fold 1 here (448px, 15 to 30 minutes on a
6 GB GPU). Demo numbers are never quoted.


In [4]:
if DEMO:
    fm_demo = rf.run_fold(1, epochs=DEMO_EPOCHS, demo=True)
else:
    print("DEMO = False — skipping the live run (definitive results load in \u00a75).")

DEMO = False — skipping the live run (definitive results load in §5).


## 4. The full five-fold run

Set `FULL_RUN = True` to run all five folds at the input size set in the config cell.
Finished folds are skipped, so the cell can be interrupted and re-run. Results are written
to `models/quality_resnet_cvF/cv_metrics.json`.


In [5]:
if FULL_RUN:
    for k in range(1, 6):
        rf.run_fold(k)
    summary = rf.aggregate()
else:
    print("FULL_RUN = False — skipping (stored results, if any, load in \u00a75).")

fold 1: train 4558 rows (699 human, 3859 machine) | test 148 human rows from 8 unseen donors
preloading 5681 slides at 448px (~3.4 GB RAM)...
  preloaded 100/5681 slides
  preloaded 200/5681 slides
  preloaded 300/5681 slides
  preloaded 400/5681 slides
  preloaded 500/5681 slides
  preloaded 600/5681 slides
  preloaded 700/5681 slides
  preloaded 800/5681 slides
  preloaded 900/5681 slides
  preloaded 1000/5681 slides
  preloaded 1100/5681 slides
  preloaded 1200/5681 slides
  preloaded 1300/5681 slides
  preloaded 1400/5681 slides
  preloaded 1500/5681 slides
  preloaded 1600/5681 slides
  preloaded 1700/5681 slides
  preloaded 1800/5681 slides
  preloaded 1900/5681 slides
  preloaded 2000/5681 slides
  preloaded 2100/5681 slides
  preloaded 2200/5681 slides
  preloaded 2300/5681 slides
  preloaded 2400/5681 slides
  preloaded 2500/5681 slides
  preloaded 2600/5681 slides
  preloaded 2700/5681 slides
  preloaded 2800/5681 slides
  preloaded 2900/5681 slides
  preloaded 3000/5681 slid

    epoch  1  train loss 0.494 F1 0.760 | val macro-F1 0.628  (best 0.628)
    epoch  2  train loss 0.289 F1 0.893 | val macro-F1 0.637  (best 0.637)
    epoch  3  train loss 0.231 F1 0.918 | val macro-F1 0.601  (best 0.637)
    epoch  4  train loss 0.217 F1 0.930 | val macro-F1 0.712  (best 0.712)
    epoch  5  train loss 0.184 F1 0.949 | val macro-F1 0.701  (best 0.712)
    epoch  6  train loss 0.172 F1 0.956 | val macro-F1 0.676  (best 0.712)
    epoch  7  train loss 0.163 F1 0.957 | val macro-F1 0.694  (best 0.712)
    epoch  8  train loss 0.150 F1 0.966 | val macro-F1 0.667  (best 0.712)
    epoch  9  train loss 0.143 F1 0.967 | val macro-F1 0.659  (best 0.712)
    epoch 10  train loss 0.138 F1 0.972 | val macro-F1 0.710  (best 0.712)
    early stop at epoch 10 (no val gain for 6)
    stage2: fine-tune on 398 human rows (val 229)
    s2 epoch  1  train loss 0.267 F1 0.894 | val macro-F1 0.837  (best 0.837)
    s2 epoch  2  train loss 0.195 F1 0.912 | val macro-F1 0.800  (best 0.83

## 5. Results of the stored run (448px)

Confusion matrices shaded by row fraction, with count and row percentage in each cell.
The dissertation quotes the 640px run (macro-F1 0.7775), not these numbers.


In [6]:
def plot_confusion_rownorm(cm, title, ax, classes=("unusable", "usable")):
    cm = np.asarray(cm, dtype=int)
    frac = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(frac, cmap="Reds", vmin=0.0, vmax=1.0)
    ax.set_xticks(range(2), classes, fontsize=9)
    ax.set_yticks(range(2), classes, fontsize=9)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n({frac[i, j] * 100:.0f}%)", ha="center",
                    va="center", fontsize=11, linespacing=1.2,
                    color="white" if frac[i, j] > 0.5 else VAL_COLOR)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("predicted", fontsize=9)
    ax.set_ylabel("true", fontsize=9)

cvf_path = p("models", "quality_resnet_cvF", "cv_metrics.json")
cvf = json.load(open(cvf_path, encoding="utf-8")) if os.path.exists(cvf_path) else None
if cvf is None:
    print("no definitive run stored yet — set FULL_RUN = True in the config cell and run \u00a74.")
else:
    a = cvf["aggregate"]
    print(f"{'fold':>4s} {'macro-F1':>9s} {'tuned':>7s} {'unus.recall':>12s} {'AUC':>7s} {'n':>4s}")
    for f in cvf["per_fold"]:
        print(f"{f['fold']:4d} {f['macro_f1']:9.4f} {f['tuned']['macro_f1']:7.4f} "
              f"{f['recall']['unusable']:12.3f} {f['auc']:7.4f} {f['n_test']:4d}")
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.6))
    plot_confusion_rownorm(a["confusion_summed"], "DEFINITIVE — summed over 5 folds (argmax)", axes[0])
    plot_confusion_rownorm(a["confusion_summed_tuned"], "DEFINITIVE — summed, val-tuned thresholds", axes[1])
    fig.tight_layout(); plt.show()
    metrics_report("ResNet-50 on Arm F combined data (DEFINITIVE, donor-grouped CV)",
        macro_f1=f"{a['macro_f1'][0]} +/- {a['macro_f1'][1]}",
        tuned_macro_f1=f"{a['tuned_macro_f1'][0]} +/- {a['tuned_macro_f1'][1]}",
        auc=a["auc"][0], recall_unusable=a["recall_unusable"][0],
        accuracy=f"{a['accuracy'][0]} (tuned {a['tuned_accuracy'][0]})",
        evaluated_on="all 847 expert labels, donor-held-out")

fold  macro-F1   tuned  unus.recall     AUC    n
   1    0.7522  0.7535        0.750  0.8165  148
   2    0.6845  0.6820        0.686  0.8360  220
   3    0.7994  0.8147        0.852  0.9277  153
   4    0.7363  0.6284        0.667  0.8538  151
   5    0.7582  0.7305        0.792  0.9019  175
METRICS FOR REPORT - ResNet-50 on Arm F combined data (DEFINITIVE, donor-grouped CV)
  macro_f1                           0.7461 +/- 0.0372
  tuned_macro_f1                     0.7218 +/- 0.0633
  auc                                0.8672
  recall_unusable                    0.7492
  accuracy                           0.8206 (tuned 0.8267)
  evaluated_on                       all 847 expert labels, donor-held-out


## 6. Paired comparison with the human-only run

Both runs share the fold partition, so the comparison is paired per fold. Argmax is
compared with argmax and tuned with tuned, never across the two. This compares the 448px
reproduction with the 640px human-only run, so part of the gap is resolution.


In [7]:
if cvf is None:
    print("comparison appears here once the definitive run exists.")
else:
    a = cvf["aggregate"]
    armb_arg = cvf["armB_reference"]["argmax_macro_f1"]
    armb_tun = cvf["armB_reference"]["tuned_macro_f1"]
    print(f"{'':22s} {'argmax':>16s} {'tuned':>16s}")
    print(f"{'Arm B (human only)':22s} {np.mean(armb_arg):16.4f} {np.mean(armb_tun):16.4f}")
    print(f"{'this run (combined)':22s} {a['macro_f1'][0]:16.4f} {a['tuned_macro_f1'][0]:16.4f}")
    print(f"{'mean paired delta':22s} {a['mean_delta_vs_armB_argmax']:+16.4f} "
          f"{a['mean_delta_vs_armB_tuned']:+16.4f}")
    x = np.arange(5)
    fig, ax = plt.subplots(figsize=(8.4, 3.4))
    ax.bar(x - 0.2, armb_arg, 0.4, color=VAL_COLOR, label="Arm B (human only)")
    ax.bar(x + 0.2, [f["macro_f1"] for f in cvf["per_fold"]], 0.4, color=TRAIN_COLOR,
           edgecolor=VAL_COLOR, label="ResNet-50 on combined data")
    ax.set_xticks(x, [f"fold {k}" for k in range(1, 6)], fontsize=9)
    ax.set_title("per-fold macro-F1, argmax — same test donors in every pair", fontsize=10)
    ax.legend(fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); plt.show()

                                 argmax            tuned
Arm B (human only)               0.8041           0.8199
this run (combined)              0.7461           0.7218
mean paired delta               -0.0580          -0.0981


## 7. Check against the research log

Every headline number is recomputed from the files on disk and compared with
`research/LOG.md`. A tick means they agree.


In [8]:
verify("combined rows", c["rows"], 5681, source="LOG 2026-08-28")
verify("human rows", c["human"], 847, source="LOG 2026-08-28")
verify("machine rows", c["machine"], 4834, source="LOG 2026-08-28")
verify("unusable rows", c["unusable"], 340, source="LOG 2026-08-28")
verify("donors", c["donors"], 43, source="LOG 2026-08-28")
verify("fold test rows sum", sum(f["n_test_human"] for f in part), 847,
       source="donor-grouped protocol")
armb = json.load(open(p("models", "quality_union_humanonly", "quality_metrics.json"),
                      encoding="utf-8"))
verify("Arm B argmax macro-F1 (re-read)",
       armb["cnn_aggregate"]["macro_f1_mean"], 0.8041, source="LOG 2026-08-22")
verify("Arm B tuned macro-F1 (re-read)",
       armb["tuned_aggregate"]["macro_f1_mean"], 0.8199, source="LOG 2026-09-01 correction")
if cvf is not None:
    pf = cvf["per_fold"]
    verify("definitive evaluated on", int(sum(f["n_test"] for f in pf)), 847,
           source="all 847, once each")
    verify("aggregate macro-F1 (re-averaged from folds)",
           round(float(np.mean([f["macro_f1"] for f in pf])), 4),
           cvf["aggregate"]["macro_f1"][0], source="cv_metrics.json internal consistency")
verify_summary()

  ✓  combined rows: computed 5681  expected 5681   [LOG 2026-08-28]
  ✓  human rows: computed 847  expected 847   [LOG 2026-08-28]
  ✓  machine rows: computed 4834  expected 4834   [LOG 2026-08-28]
  ✓  unusable rows: computed 340  expected 340   [LOG 2026-08-28]
  ✓  donors: computed 43  expected 43   [LOG 2026-08-28]
  ✓  fold test rows sum: computed 847  expected 847   [donor-grouped protocol]
  ✓  Arm B argmax macro-F1 (re-read): computed 0.8041  expected 0.8041   [LOG 2026-08-22]
  ✓  Arm B tuned macro-F1 (re-read): computed 0.8199  expected 0.8199   [LOG 2026-09-01 correction]
  ✓  definitive evaluated on: computed 847  expected 847   [all 847, once each]
  ✓  aggregate macro-F1 (re-averaged from folds): computed 0.7461  expected 0.7461   [cv_metrics.json internal consistency]

VERIFICATION: 10/10 checks passed
